# SPX option data preprocessing

이 노트북은 `01_01_option_chain_using_API.ipynb`에서 시작한 `download_spx_chain`을 패키지에서 import하여, 고정된 historical SPX option-chain snapshot을 한 번 수집하고 분석용 pandas 데이터로 전처리한다. 기존 API 노트북 전체를 `%run`하지 않으므로 과거 요청 셀은 실행되지 않는다.

## 1. 고정 수집 조건과 raw cache 정책

수집 조건은 `date=2026-07-15`, `dte=30`, `am=false`, `pm=true`, `strikeLimit=450`으로 고정한다. 예상 raw CSV가 이미 존재하고 비어 있지 않으면 그 파일만 읽는다. 경로가 없을 때만 API를 한 번 호출하며, 실패해도 날짜·DTE·strike limit을 바꾸거나 자동 재시도하지 않는다. 빈 raw 파일은 덮어쓰지 않고 blocker로 취급한다.

In [1]:
from __future__ import annotations

import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
)
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from option_pricing_volatility.market_data import download_spx_chain

QUOTE_DATE = "2026-07-15"
TARGET_DTE = 30
STRIKE_LIMIT = 450

STEM = "SPX_2026-07-15_dte030_pm_sl450"
RAW_DIR = PROJECT_ROOT / "data/raw/marketdata_spx"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/marketdata_spx"
INTERIM_DIR = PROJECT_ROOT / "data/interim/marketdata_spx"
RAW_PATH = RAW_DIR / f"{STEM}.csv"
PROCESSED_PATH = PROCESSED_DIR / f"{STEM}_processed.csv"
REJECTED_PATH = INTERIM_DIR / f"{STEM}_rejected.csv"

print("Project root resolved from Git.")
print(f"Expected raw snapshot: {RAW_PATH.relative_to(PROJECT_ROOT)}")

Project root resolved from Git.
Expected raw snapshot: data/raw/marketdata_spx/SPX_2026-07-15_dte030_pm_sl450.csv


In [2]:
raw_was_cached = RAW_PATH.exists() and RAW_PATH.stat().st_size > 0
raw_df = download_spx_chain(
    quote_date=QUOTE_DATE,
    target_dte=TARGET_DTE,
    strike_limit=STRIKE_LIMIT,
    raw_dir=RAW_DIR,
)
if raw_df.empty:
    raise RuntimeError("The raw SPX snapshot contains no option rows")

acquisition_source = "cache" if raw_was_cached else "single API request"
print(f"Acquisition source: {acquisition_source}")
print(f"Raw shape: {raw_df.shape}")

[CACHE] Reading existing raw snapshot: SPX_2026-07-15_dte030_pm_sl450.csv
Acquisition source: cache
Raw shape: (430, 27)


## 2. 전처리 범위와 금융 해석

`updated`를 실제 quote 시각, `expiration`을 실제 만기 시각으로 사용하고 둘 다 UTC-aware timestamp로 변환한다. 잔존만기 `T`는 두 시각의 실제 초 차이를 **ACT/365F**로 연환산한다.

현재 `spot_moneyness = strike / spot`과 0.85–1.15 후보 범위는 향후 분석을 위한 임시 spot 기준이다. 이 단계에는 외부 입력이 필요한 `risk_free_rate`, `dividend_yield`, `forward`, `log_forward_moneyness`를 만들거나 추정하지 않는다. MarketData의 `vendor_iv`는 원 값을 보존하는 참고 필드이며 프로젝트가 자체 계산한 IV로 취급하지 않는다. `volume == 0` 또는 `open_interest == 0`만으로는 행을 제거하지 않는다.

In [3]:
def to_snake_case(name: object) -> str:
    """Convert a vendor column name to a stable snake_case label."""
    text = re.sub(r"[^0-9A-Za-z]+", "_", str(name).strip())
    text = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", text)
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", text)
    return text.strip("_").lower()


def blank_mask(series: pd.Series) -> pd.Series:
    """Identify null or whitespace-only required values."""
    return series.isna() | series.astype("string").str.strip().eq("").fillna(False)


working_df = raw_df.copy(deep=True).reset_index(drop=True)
snake_columns = [to_snake_case(column) for column in working_df.columns]
duplicate_columns = pd.Index(snake_columns)[pd.Index(snake_columns).duplicated()].unique().tolist()
if duplicate_columns:
    raise ValueError(f"snake_case column collision: {duplicate_columns}")
working_df.columns = snake_columns
working_df = working_df.rename(
    columns={
        "side": "option_type",
        "updated": "quote_timestamp",
        "expiration": "expiry_timestamp",
        "underlying_price": "spot",
        "iv": "vendor_iv",
    }
)
working_df.insert(0, "raw_row_number", np.arange(len(working_df), dtype=int))

required_text_columns = ("option_symbol", "option_type")
required_numeric_columns = ("strike", "bid", "ask", "spot")
required_timestamp_columns = ("quote_timestamp", "expiry_timestamp")
for column in (*required_text_columns, *required_numeric_columns, *required_timestamp_columns):
    if column not in working_df.columns:
        working_df[column] = pd.NA

reasons: list[list[str]] = [[] for _ in range(len(working_df))]


def add_reason(mask: pd.Series, reason: str) -> None:
    """Append one deterministic reason code to every selected raw row."""
    selected = np.flatnonzero(mask.fillna(False).to_numpy(dtype=bool))
    for position in selected:
        reasons[position].append(reason)


for column in required_text_columns:
    original = working_df[column]
    missing = blank_mask(original)
    normalized = original.astype("string").str.strip()
    if column == "option_type":
        normalized = normalized.str.lower()
    working_df[column] = normalized.mask(missing, pd.NA)
    add_reason(missing, f"MISSING_REQUIRED_FIELD:{column}")

for column in required_numeric_columns:
    original = working_df[column]
    missing = blank_mask(original)
    numeric = pd.to_numeric(original, errors="coerce")
    non_numeric = ~missing & numeric.isna()
    nonfinite = numeric.notna() & ~np.isfinite(numeric)
    working_df[column] = numeric
    add_reason(missing, f"MISSING_REQUIRED_FIELD:{column}")
    add_reason(non_numeric, f"NON_NUMERIC_REQUIRED_FIELD:{column}")
    add_reason(nonfinite, f"NONFINITE_REQUIRED_FIELD:{column}")

for column in required_timestamp_columns:
    original = working_df[column]
    missing = blank_mask(original)
    epoch_seconds = pd.to_numeric(original, errors="coerce")
    non_numeric = ~missing & epoch_seconds.isna()
    nonfinite = epoch_seconds.notna() & ~np.isfinite(epoch_seconds)
    converted = pd.to_datetime(
        epoch_seconds.mask(nonfinite),
        unit="s",
        utc=True,
        errors="coerce",
    )
    invalid_timestamp = ~missing & ~non_numeric & ~nonfinite & converted.isna()
    working_df[column] = converted
    add_reason(missing, f"MISSING_REQUIRED_FIELD:{column}")
    add_reason(non_numeric, f"NON_NUMERIC_REQUIRED_FIELD:{column}")
    add_reason(nonfinite, f"NONFINITE_REQUIRED_FIELD:{column}")
    add_reason(invalid_timestamp, f"INVALID_TIMESTAMP:{column}")

optional_numeric_columns = (
    "vendor_iv", "volume", "open_interest", "dte", "requested_dte",
    "bid_size", "ask_size", "last", "intrinsic_value",
    "extrinsic_value", "delta", "gamma", "theta", "vega",
)
for column in optional_numeric_columns:
    if column in working_df.columns:
        working_df[column] = pd.to_numeric(working_df[column], errors="coerce")

if "first_traded" in working_df.columns:
    first_traded_seconds = pd.to_numeric(working_df["first_traded"], errors="coerce")
    working_df["first_traded"] = pd.to_datetime(
        first_traded_seconds.where(np.isfinite(first_traded_seconds)),
        unit="s",
        utc=True,
        errors="coerce",
    )

working_df["mid"] = (working_df["bid"] + working_df["ask"]) / 2.0
working_df["spread"] = working_df["ask"] - working_df["bid"]
valid_mid = working_df["mid"].gt(0) & np.isfinite(working_df["mid"])
working_df["relative_spread"] = (working_df["spread"] / working_df["mid"]).where(valid_mid)
working_df["T"] = (
    (working_df["expiry_timestamp"] - working_df["quote_timestamp"]).dt.total_seconds()
    / (365.0 * 24.0 * 60.0 * 60.0)
)
valid_moneyness_inputs = (
    working_df["strike"].gt(0)
    & working_df["spot"].gt(0)
    & np.isfinite(working_df["strike"])
    & np.isfinite(working_df["spot"])
)
working_df["spot_moneyness"] = (working_df["strike"] / working_df["spot"]).where(valid_moneyness_inputs)
working_df["log_spot_moneyness"] = np.log(working_df["spot_moneyness"])
working_df["analysis_candidate"] = (
    working_df["spot_moneyness"].between(0.85, 1.15, inclusive="both").fillna(False)
)

valid_option_type = working_df["option_type"].isin(["call", "put"])
add_reason(working_df["option_type"].notna() & ~valid_option_type, "INVALID_OPTION_TYPE")
add_reason(working_df["strike"].notna() & working_df["strike"].le(0), "NONPOSITIVE_STRIKE")
add_reason(working_df["spot"].notna() & working_df["spot"].le(0), "NONPOSITIVE_SPOT")
add_reason(working_df["bid"].notna() & working_df["bid"].le(0), "NONPOSITIVE_BID")
add_reason(working_df["ask"].notna() & working_df["bid"].notna() & working_df["ask"].lt(working_df["bid"]), "CROSSED_QUOTE")
add_reason(working_df["mid"].notna() & np.isfinite(working_df["mid"]) & working_df["mid"].le(0), "NONPOSITIVE_MID")
add_reason(working_df["T"].notna() & np.isfinite(working_df["T"]) & working_df["T"].le(0), "NONPOSITIVE_TTM")
duplicate_symbol = working_df["option_symbol"].notna() & working_df["option_symbol"].duplicated(keep=False)
add_reason(duplicate_symbol, "DUPLICATE_OPTION_SYMBOL")

working_df["rejection_reason"] = ["|".join(dict.fromkeys(row_reasons)) for row_reasons in reasons]
core_columns = [
    "raw_row_number", "quote_date", "option_symbol", "underlying",
    "option_type", "strike", "quote_timestamp", "expiry_timestamp",
    "T", "spot", "bid", "ask", "mid", "spread",
    "relative_spread", "spot_moneyness", "log_spot_moneyness",
    "analysis_candidate", "vendor_iv", "volume", "open_interest",
]
ordered_columns = [column for column in core_columns if column in working_df.columns]
ordered_columns += [column for column in working_df.columns if column not in ordered_columns]
working_df = working_df.loc[:, ordered_columns]

accepted_mask = working_df["rejection_reason"].eq("")
accepted_df = working_df.loc[accepted_mask].drop(columns="rejection_reason").reset_index(drop=True)
rejected_df = working_df.loc[~accepted_mask].reset_index(drop=True)

assert len(raw_df) == len(accepted_df) + len(rejected_df)
print(f"Accepted rows: {len(accepted_df):,}")
print(f"Rejected rows: {len(rejected_df):,}")

Accepted rows: 424
Rejected rows: 6


In [4]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
accepted_df.to_csv(PROCESSED_PATH, index=False)
rejected_df.to_csv(REJECTED_PATH, index=False)

print(f"Saved processed data: {PROCESSED_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved rejected data: {REJECTED_PATH.relative_to(PROJECT_ROOT)}")

Saved processed data: data/processed/marketdata_spx/SPX_2026-07-15_dte030_pm_sl450_processed.csv
Saved rejected data: data/interim/marketdata_spx/SPX_2026-07-15_dte030_pm_sl450_rejected.csv


## 3. Snapshot 감사

마지막 셀은 행 보존, 유효 quote 조건, 실제 만기, 파생값 범위, 거절 사유와 저장 후 재로딩 행 수를 한 번에 출력하고 assertion으로 검증한다.

In [5]:
processed_reload = pd.read_csv(PROCESSED_PATH)
rejected_reload = pd.read_csv(REJECTED_PATH)

raw_count = len(raw_df)
accepted_count = len(accepted_df)
rejected_count = len(rejected_df)
candidate_count = int(accepted_df["analysis_candidate"].sum())
partition_ok = raw_count == accepted_count + rejected_count
call_put_counts = accepted_df["option_type"].value_counts().reindex(["call", "put"], fill_value=0)
expiry_values = (
    accepted_df["expiry_timestamp"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    .tolist()
)
unique_strike_count = accepted_df["strike"].nunique(dropna=True)
moneyness_range = (accepted_df["spot_moneyness"].min(), accepted_df["spot_moneyness"].max())
relative_spread_quantiles = accepted_df["relative_spread"].quantile([0.0, 0.25, 0.5, 0.75, 1.0])
required_output_columns = [
    "option_symbol", "option_type", "strike", "spot", "bid",
    "ask", "mid", "spread", "relative_spread",
    "quote_timestamp", "expiry_timestamp", "T",
    "spot_moneyness", "log_spot_moneyness", "analysis_candidate",
]
required_missing = accepted_df[required_output_columns].isna().sum()
raw_duplicate_symbol_rows = int(working_df["option_symbol"].notna().mul(working_df["option_symbol"].duplicated(keep=False)).sum())
accepted_duplicate_symbol_rows = int(accepted_df["option_symbol"].duplicated(keep=False).sum())
if rejected_df.empty:
    rejection_reason_counts = pd.Series(dtype="int64", name="count")
else:
    rejection_reason_counts = (
        rejected_df["rejection_reason"]
        .str.split(r"\|", regex=True)
        .explode()
        .value_counts()
        .sort_index()
    )

print("Row counts:", {"raw": raw_count, "accepted": accepted_count, "rejected": rejected_count, "analysis_candidate": candidate_count})
print("raw = accepted + rejected:", partition_ok)
print("Accepted call/put counts:\n", call_put_counts.to_string())
print(f"Actual expiries ({len(expiry_values)}): {expiry_values}")
print("Unique accepted strikes:", unique_strike_count)
print("Spot moneyness min/max:", tuple(round(float(value), 8) for value in moneyness_range))
print("Relative spread quantiles:\n", relative_spread_quantiles.to_string())
print("Required-column missing values:\n", required_missing.to_string())
print("Duplicate option-symbol rows:", {"raw": raw_duplicate_symbol_rows, "accepted": accepted_duplicate_symbol_rows})
print("Rejection reason counts:\n", rejection_reason_counts.to_string())
print("Reloaded row counts:", {"processed": len(processed_reload), "rejected": len(rejected_reload)})

accepted_ids = set(accepted_df["raw_row_number"])
rejected_ids = set(rejected_df["raw_row_number"])
assert partition_ok
assert accepted_ids.isdisjoint(rejected_ids)
assert accepted_ids | rejected_ids == set(range(raw_count))
assert accepted_count > 0
assert required_missing.eq(0).all()
assert accepted_df["option_type"].isin(["call", "put"]).all()
assert accepted_df["strike"].gt(0).all()
assert accepted_df["spot"].gt(0).all()
assert accepted_df["bid"].gt(0).all()
assert accepted_df["ask"].ge(accepted_df["bid"]).all()
assert accepted_df["mid"].gt(0).all()
assert accepted_df["T"].gt(0).all()
assert np.isfinite(accepted_df[["mid", "spread", "relative_spread", "T", "spot_moneyness", "log_spot_moneyness"]].to_numpy()).all()
assert accepted_duplicate_symbol_rows == 0
assert len(processed_reload) == accepted_count
assert len(rejected_reload) == rejected_count
assert {"risk_free_rate", "dividend_yield", "forward", "log_forward_moneyness"}.isdisjoint(accepted_df.columns)
print("Audit assertions: PASS")

Row counts: {'raw': 430, 'accepted': 424, 'rejected': 6, 'analysis_candidate': 352}
raw = accepted + rejected: True
Accepted call/put counts:
 option_type
call    213
put     211
Actual expiries (1): ['2026-08-14T20:00:00Z']
Unique accepted strikes: 215
Spot moneyness min/max: (0.31694004, 1.29417183)
Relative spread quantiles:
 0.00    0.002759
0.25    0.007467
0.50    0.015454
0.75    0.026966
1.00    1.000000
Required-column missing values:
 option_symbol         0
option_type           0
strike                0
spot                  0
bid                   0
ask                   0
mid                   0
spread                0
relative_spread       0
quote_timestamp       0
expiry_timestamp      0
T                     0
spot_moneyness        0
log_spot_moneyness    0
analysis_candidate    0
Duplicate option-symbol rows: {'raw': 0, 'accepted': 0}
Rejection reason counts:
 rejection_reason
NONPOSITIVE_BID    6
Reloaded row counts: {'processed': 424, 'rejected': 6}
Audit assertions